In [51]:
import itertools
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

pd.set_option("display.max_colwidth", 200)

In [52]:
KEYWORD_CONFIG = {
    "object_categories": {
        "consumer_electronics": [
            "airpods", "earbuds", "headphones", "ipad", "tablet", "macbook"
        ],
        "camera_drone": [
            "camera", "camera lens", "drone", "gopro", "drone battery"
        ],
        "digital_accessories": [
            "charger", "cables", "power bank", "adapter", "usb hub", "ssd", "hard drive"
        ],
        "gaming": [
            "switch", "steam deck", "keyboard", "gaming gear"
        ],
        "valuable_items": [
            "watches", "jewelry", "collectibles", "figurines", "cards", "makeup"
        ],
        "travel_outdoor": [
            "travel gear", "camping gear", "hiking equipment", "carry on essentials", "tools"
        ]
    },

    "intents": [
        "case",
        "hard case",
        "protective case",
        "storage",
        "organizer",
        "holder",
        "box",
        "carrying case",
        "travel case"
    ],

    "scenarios": [
        "review",
        "unboxing",
        "setup",
        "how to use",
        "how to carry",
        "how to store",
        "travel",
        "daily use",
        "EDC",
        "desk setup"
    ],

    "pain_patterns": [
        "how to protect {obj}",
        "how to store {obj}",
        "best way to carry {obj}",
        "safe storage for {obj}",
        "{obj} scratched easily",
        "{obj} easy to break",
        "{obj} needs a case",
        "no good case for {obj}",
        "best hard case for {obj}"
    ],

    "global_demand_signals": [
        "wish it had a case",
        "needs a case",
        "needs a hard case",
        "no protection",
        "not protected",
        "easy to break",
        "fragile",
        "scratched easily",
        "hard to carry",
        "no storage",
        "messy cables"
    ],

    "global_trends": [
        "tech accessories 2025",
        "new gadgets 2025",
        "viral gadgets",
        "amazon finds tech",
        "tiktok gadgets"
    ]
}

In [53]:
config_dir = Path("../../data/shared_data/config")
config_dir.mkdir(parents=True, exist_ok=True)

config_file = config_dir / "keyword_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(KEYWORD_CONFIG, f, ensure_ascii=False, indent=2)

print("配置已保存到：", config_file)

配置已保存到： ../../data/shared_data/config/keyword_config.json


In [54]:
def load_keyword_config(config_path: Path) -> dict:
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)

config = load_keyword_config(config_file)

print(config.keys())

dict_keys(['object_categories', 'intents', 'scenarios', 'pain_patterns', 'global_demand_signals', 'global_trends'])


In [55]:
def generate_queries_for_categories(
    config: dict,
    categories: list[str],
    include_global_terms: bool = False
) -> pd.DataFrame:
    rows = []

    intents = config["intents"]
    scenarios = config["scenarios"]
    pain_patterns = config["pain_patterns"]

    for category in categories:
        objects = config["object_categories"].get(category, [])

        for obj, intent, scenario in itertools.product(objects, intents, scenarios):
            rows.append({
                "category": category,
                "object": obj,
                "intent": intent,
                "scenario": scenario,
                "pain_pattern": None,
                "query": f"{obj} {intent} {scenario}",
                "query_type": "product_intent_scenario"
            })

        for obj, intent in itertools.product(objects, intents):
            rows.append({
                "category": category,
                "object": obj,
                "intent": intent,
                "scenario": None,
                "pain_pattern": None,
                "query": f"{obj} {intent}",
                "query_type": "product_intent"
            })

        for obj, pattern in itertools.product(objects, pain_patterns):
            rows.append({
                "category": category,
                "object": obj,
                "intent": None,
                "scenario": None,
                "pain_pattern": pattern,
                "query": pattern.format(obj=obj),
                "query_type": "pain_or_howto"
            })

    if include_global_terms:
        for signal in config["global_demand_signals"]:
            rows.append({
                "category": "global",
                "object": None,
                "intent": None,
                "scenario": None,
                "pain_pattern": None,
                "query": signal,
                "query_type": "global_demand_signal"
            })

        for trend in config["global_trends"]:
            rows.append({
                "category": "global",
                "object": None,
                "intent": None,
                "scenario": None,
                "pain_pattern": None,
                "query": trend,
                "query_type": "global_trend"
            })

    df = pd.DataFrame(rows).drop_duplicates(subset=["query"]).reset_index(drop=True)
    return df

In [56]:
df_test = generate_queries_for_categories(
    config=config,
    categories=["camera_drone", "gaming"],
    include_global_terms=False
)

print("query 数量：", len(df_test))
df_test.head(20)

query 数量： 972


,category,object,intent,scenario,pain_pattern,query,query_type
0,camera_drone,camera,case,review,None,camera case review,product_intent_scenario
1,camera_drone,camera,case,unboxing,None,camera case unboxing,product_intent_scenario
2,camera_drone,camera,case,setup,None,camera case setup,product_intent_scenario
3,camera_drone,camera,case,how to use,None,camera case how to use,product_intent_scenario
4,camera_drone,camera,case,how to carry,None,camera case how to carry,product_intent_scenario
5,camera_drone,camera,case,how to store,None,camera case how to store,product_intent_scenario
6,camera_drone,camera,case,travel,None,camera case travel,product_intent_scenario
7,camera_drone,camera,case,daily use,None,camera case daily use,product_intent_scenario
8,camera_drone,camera,case,EDC,None,camera case EDC,product_intent_scenario
9,camera_drone,camera,case,desk setup,None,camera case desk setup,product_intent_scenario


In [57]:
def stratified_sample_queries(df: pd.DataFrame, sample_plan: dict, random_seed: int = 42) -> pd.DataFrame:
    sampled_parts = []

    for qtype, n in sample_plan.items():
        sub = df[df["query_type"] == qtype]
        if len(sub) == 0:
            continue
        sampled_parts.append(sub.sample(n=min(n, len(sub)), random_state=random_seed))

    if not sampled_parts:
        return pd.DataFrame(columns=df.columns)

    out = pd.concat(sampled_parts, ignore_index=True).drop_duplicates(subset=["query"]).reset_index(drop=True)
    return out

In [58]:
MEMBER_ASSIGNMENT = {
    "member_1": ["consumer_electronics"],
    "member_2": ["camera_drone"],
    "member_3": ["digital_accessories"],
    "member_4": ["gaming", "valuable_items"],
    "member_5": ["travel_outdoor"]
}

In [59]:
SAMPLE_PLAN = {
    "product_intent_scenario": 60,
    "product_intent": 30,
    "pain_or_howto": 35
}

GLOBAL_SAMPLE_PLAN = {
    "global_demand_signal": 11,
    "global_trend": 5
}

In [ ]:
output_dir = Path("../data/shared_data/query_assignments")
output_dir.mkdir(parents=True, exist_ok=True)

today_str = datetime.now().strftime("%Y%m%d")

all_assignment_tables = []

for member_name, categories in MEMBER_ASSIGNMENT.items():
    df_member_full = generate_queries_for_categories(
        config=config,
        categories=categories,
        include_global_terms=False
    )

    df_member_sampled = stratified_sample_queries(
        df=df_member_full,
        sample_plan=SAMPLE_PLAN,
        random_seed=42
    ).copy()

    df_member_sampled["assigned_to"] = member_name

    file_path = output_dir / f"{member_name}_queries_{today_str}.csv"
    df_member_sampled.to_csv(file_path, index=False, encoding="utf-8-sig")

    all_assignment_tables.append(df_member_sampled)

    print(f"已生成：{file_path} | {len(df_member_sampled)} 条")

已生成：../data/shared_data/query_assignments/member_1_queries_20260330.csv | 125 条
已生成：../data/shared_data/query_assignments/member_2_queries_20260330.csv | 125 条
已生成：../data/shared_data/query_assignments/member_3_queries_20260330.csv | 125 条
已生成：../data/shared_data/query_assignments/member_4_queries_20260330.csv | 125 条
已生成：../data/shared_data/query_assignments/member_5_queries_20260330.csv | 125 条


In [61]:
df_global_full = generate_queries_for_categories(
    config=config,
    categories=[],
    include_global_terms=True
)

df_global = stratified_sample_queries(
    df=df_global_full,
    sample_plan=GLOBAL_SAMPLE_PLAN,
    random_seed=42
).copy()

df_global["assigned_to"] = "leader"

global_file = output_dir / f"leader_global_queries_{today_str}.csv"
df_global.to_csv(global_file, index=False, encoding="utf-8-sig")

print(f"已生成：{global_file} | {len(df_global)} 条")
df_global.head()

已生成：../data/shared_data/query_assignments/leader_global_queries_20260330.csv | 16 条


,category,object,intent,scenario,pain_pattern,query,query_type,assigned_to
0,global,None,None,None,None,easy to break,global_demand_signal,leader
1,global,None,None,None,None,wish it had a case,global_demand_signal,leader
2,global,None,None,None,None,no storage,global_demand_signal,leader
3,global,None,None,None,None,messy cables,global_demand_signal,leader
4,global,None,None,None,None,needs a hard case,global_demand_signal,leader


In [62]:
df_all_members = pd.concat(all_assignment_tables, ignore_index=True)
df_master = pd.concat([df_all_members, df_global], ignore_index=True)

master_file = output_dir / f"ALL_QUERY_ASSIGNMENTS_{today_str}.csv"
df_master.to_csv(master_file, index=False, encoding="utf-8-sig")

print("总表已导出：", master_file)
print("总 query 数：", len(df_master))
df_master.head(20)

总表已导出： ../data/shared_data/query_assignments/ALL_QUERY_ASSIGNMENTS_20260330.csv
总 query 数： 641


,category,object,intent,scenario,pain_pattern,query,query_type,assigned_to
0,consumer_electronics,headphones,organizer,desk setup,None,headphones organizer desk setup,product_intent_scenario,member_1
1,consumer_electronics,airpods,carrying case,how to use,None,airpods carrying case how to use,product_intent_scenario,member_1
2,consumer_electronics,macbook,carrying case,unboxing,None,macbook carrying case unboxing,product_intent_scenario,member_1
3,consumer_electronics,airpods,travel case,travel,None,airpods travel case travel,product_intent_scenario,member_1
4,consumer_electronics,macbook,hard case,desk setup,None,macbook hard case desk setup,product_intent_scenario,member_1
5,consumer_electronics,airpods,carrying case,daily use,None,airpods carrying case daily use,product_intent_scenario,member_1
6,consumer_electronics,ipad,protective case,daily use,None,ipad protective case daily use,product_intent_scenario,member_1
7,consumer_electronics,tablet,holder,EDC,None,tablet holder EDC,product_intent_scenario,member_1
8,consumer_electronics,ipad,carrying case,setup,None,ipad carrying case setup,product_intent_scenario,member_1
9,consumer_electronics,tablet,protective case,how to use,None,tablet protective case how to use,product_intent_scenario,member_1


In [63]:
dup_queries = (
    df_all_members.groupby("query")
    .agg(
        assigned_count=("assigned_to", "nunique"),
        assigned_members=("assigned_to", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

dup_queries = dup_queries[dup_queries["assigned_count"] > 1].sort_values("assigned_count", ascending=False)

print("跨组员重复 query 数量：", len(dup_queries))
dup_queries.head(20)

跨组员重复 query 数量： 0


,query,assigned_count,assigned_members


In [64]:
simple_dir = output_dir / "simple_query_only"
simple_dir.mkdir(parents=True, exist_ok=True)

for member_name, categories in MEMBER_ASSIGNMENT.items():
    src_file = output_dir / f"{member_name}_queries_{today_str}.csv"
    df = pd.read_csv(src_file)

    simple_df = df[["query"]].drop_duplicates().reset_index(drop=True)

    dst_file = simple_dir / f"{member_name}_query_only_{today_str}.csv"
    simple_df.to_csv(dst_file, index=False, encoding="utf-8-sig")

    print(f"已生成简化版：{dst_file} | {len(simple_df)} 条")

leader_simple = df_global[["query"]].drop_duplicates().reset_index(drop=True)
leader_simple_file = simple_dir / f"leader_global_query_only_{today_str}.csv"
leader_simple.to_csv(leader_simple_file, index=False, encoding="utf-8-sig")

print(f"已生成组长简化版：{leader_simple_file} | {len(leader_simple)} 条")

已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_1_query_only_20260330.csv | 125 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_2_query_only_20260330.csv | 125 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_3_query_only_20260330.csv | 125 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_4_query_only_20260330.csv | 125 条
已生成简化版：../data/shared_data/query_assignments/simple_query_only/member_5_query_only_20260330.csv | 125 条
已生成组长简化版：../data/shared_data/query_assignments/simple_query_only/leader_global_query_only_20260330.csv | 16 条
